# 🔬 Kaggle Phase 6A3 (Part 1): Ablation Experiment (Alpha = 20.0)
## Schedules Evaluated in Part 1: Continuous Full Steering, Hard Cutoff Early-Stop

**Fixed Parameter:** $\alpha = 20.0$  
**Test Set:** $N_{{test}} = 500$ independent clinical questions  
**Execution Budget:** ~5.5 hours (Prevents Kaggle 12h GPU timeout!)

In [1]:
# Cell 1: Install Dependencies
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm pandas numpy
print('✅ Dependencies installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.1 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [2]:
# Cell 2: Imports & Environment Setup
import os, json, glob, random, time, math, gc
import numpy as np, pandas as pd, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rouge_score import rouge_scorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = '/kaggle/working'
ALPHA_VAL = 20.0
suffix = '3'
part_num = 1
print(f'✅ Environment ready | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'✅ Configured Alpha Value: {ALPHA_VAL} | Suffix: {suffix} | Part: {part_num}')

✅ Environment ready | GPU: Tesla T4
✅ Configured Alpha Value: 20.0 | Suffix: 3 | Part: 1


In [3]:
# Cell 3: Robust Data & Steering Vector Loading
data_path = None
priority_filenames = ['vietnamese_medical_halueval_15k_specialized.json', 'vietnamese_medical_halueval_15k.json']
for pf in priority_filenames:
    matches = glob.glob(f'/kaggle/input/**/{pf}', recursive=True)
    if matches: data_path = matches[0]; print(f'✅ Found PRIORITY dataset: {data_path}'); break

if not data_path:
    all_jsons = glob.glob('/kaggle/input/**/*.json', recursive=True)
    for j in all_jsons:
        bname = os.path.basename(j).lower()
        if any(skip in bname for skip in ['huggingface', 'config', 'steering']): continue
        if any(kw in bname for kw in ['medical', 'halueval', 'generated', 'phase3']):
            data_path = j; print(f'⚠️ Using fallback dataset: {data_path}'); break

if not data_path: raise FileNotFoundError('❌ No medical test dataset found in /kaggle/input/')

with open(data_path, 'r', encoding='utf-8') as f: raw_dataset = json.load(f)
if isinstance(raw_dataset, dict):
    unpacked = []
    for k, v in raw_dataset.items():
        if isinstance(v, list): unpacked.extend(v)
        elif isinstance(v, dict): unpacked.append(v)
    raw_dataset = unpacked

shuffled_records = list(raw_dataset)
random.seed(SEED); random.shuffle(shuffled_records)
n_total = len(shuffled_records)
if n_total > 500:
    n_train = int(n_total * 0.70); n_val = int(n_total * 0.15)
    test_records = shuffled_records[n_train + n_val:]
else: test_records = shuffled_records

print(f'📊 Total records: {n_total:,} | Test Split: {len(test_records):,} records')
print(f'📊 Evaluating on min(500, {len(test_records)}) = {min(500, len(test_records))} test samples')

config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True)
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
if not v_steer_paths: raise FileNotFoundError('❌ v_steer.pt not found! Add steering-phase1-artifacts.')

BEST_LAYER = 8
if config_paths:
    with open(config_paths[0], 'r') as f: steering_config = json.load(f)
    BEST_LAYER = steering_config.get('best_layer', 8)

v_steer = torch.load(v_steer_paths[0], map_location='cpu')
print(f'✅ Layer: {BEST_LAYER} | v_steer shape: {v_steer.shape}')

✅ Found PRIORITY dataset: /kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json
📊 Total records: 14,700 | Test Split: 2,205 records
📊 Evaluating on min(500, 2205) = 500 test samples
✅ Layer: 8 | v_steer shape: torch.Size([3584])


In [4]:
# Cell 4: Load Qwen2.5-7B Model
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'⌛ Loading {MODEL_NAME} in 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map='auto', trust_remote_code=True
)
model.eval()
print('✅ Qwen2.5-7B-Instruct loaded successfully!')

⌛ Loading Qwen/Qwen2.5-7B-Instruct in 4-bit NF4...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen2.5-7B-Instruct loaded successfully!


In [5]:
# Cell 5: Steering Hook & Evaluation Engine (Multi-GPU Safe)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

class SteeringHook:
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=16, decay='linear'):
        self.layer_idx = layer_idx; self.v_vector = v_vector
        self.alpha = alpha; self.K = K; self.decay = decay
        self.step_counter = 0; self.handle = None

    def _eff_alpha(self, step):
        if self.decay == 'none' or self.K >= 999: return self.alpha
        if step > self.K: return 0.0
        if self.decay == 'hard': return self.alpha
        elif self.decay == 'linear': return self.alpha * (1.0 - (step - 1) / self.K)
        elif self.decay == 'matched_dose': return (self.alpha * 16.0 / step) if step <= self.K else 0.0
        return 0.0

    def hook_fn(self, module, input, output):
        if isinstance(output, tuple): h = output[0]
        else: h = output
        if h.shape[1] == 1 and self.v_vector is not None:
            self.step_counter += 1
            scale = self._eff_alpha(self.step_counter)
            if scale > 0:
                v_eff = self.v_vector.to(h.device, dtype=h.dtype)
                h[:, -1, :] = h[:, -1, :] + scale * v_eff
        if isinstance(output, tuple): return (h,) + output[1:]
        return h

    def register(self, model):
        self.handle = model.model.layers[self.layer_idx].register_forward_hook(self.hook_fn)
    def remove(self):
        if self.handle: self.handle.remove(); self.step_counter = 0

def compute_rep4(text):
    tokens = text.split()
    if len(tokens) < 4: return 0.0
    grams = [tuple(tokens[i:i+4]) for i in range(len(tokens)-3)]
    return 1.0 - (len(set(grams)) / len(grams))

def evaluate_condition(model, tokenizer, dataset, v_vector, alpha, K, decay, max_new_tokens=200, name=''):
    results = []
    for idx, item in enumerate(tqdm(dataset, desc=name)):
        if isinstance(item, str): ctx = ''; q = item; ref = item
        elif isinstance(item, dict):
            ctx = item.get('knowledge_context', item.get('context', ''))
            q = item.get('question', item.get('prompt', ''))
            ref = item.get('right_answer', item.get('reference', item.get('answer', '')))
        else: continue

        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to(device)

        hook = SteeringHook(BEST_LAYER, v_vector, alpha=alpha, K=K, decay=decay)
        if v_vector is not None: hook.register(model)

        t0 = time.time()
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
            )
        elapsed = (time.time() - t0) * 1000
        if v_vector is not None: hook.remove()

        gen_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        rg = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100 if ref else 0.0
        rep4 = compute_rep4(gen_text)
        hit_eos = 1 if (tokenizer.eos_token_id in gen_tokens.tolist()) else 0

        results.append({
            'idx': idx, 'generated': gen_text, 'reference': ref,
            'rouge_l': rg, 'rep_4gram': rep4, 'hit_eos': hit_eos,
            'num_tokens': len(gen_tokens), 'elapsed_ms': elapsed
        })
    return results

print('✅ Steering engine ready (Multi-GPU safe, max_new_tokens=200)')

✅ Steering engine ready (Multi-GPU safe, max_new_tokens=200)


In [6]:
# Cell 6: 🚀 Run Part 1 Schedules at Alpha = 20.0
TEST_LIMIT = min(500, len(test_records))
test_subset = test_records[:TEST_LIMIT]
results_dict = {}
print(f'🚀 Running Part 1 (Continuous Full Steering, Hard Cutoff Early-Stop) | Alpha = {ALPHA_VAL} | N_test = {TEST_LIMIT}')

# [1/2] Continuous Full Steering
print(f'\n--- [1/2] Continuous Full Steering (K=999, alpha={ALPHA_VAL}) ---')
results_dict['continuous'] = evaluate_condition(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_VAL, K=999, decay='none', name='Continuous Full Steering')

# [2/2] Hard Cutoff Early-Stop
print(f'\n--- [2/2] Hard Cutoff Early-Stop (K=16, alpha={ALPHA_VAL}) ---')
results_dict['hard_cutoff'] = evaluate_condition(model, tokenizer, test_subset,
    v_vector=v_steer, alpha=ALPHA_VAL, K=16, decay='hard', name='Hard Cutoff Early-Stop')

gc.collect(); torch.cuda.empty_cache()
print(f'\n✅ Part 1 (Continuous Full Steering, Hard Cutoff Early-Stop) completed for Alpha = {ALPHA_VAL}!')

🚀 Running Part 1 (Continuous Full Steering, Hard Cutoff Early-Stop) | Alpha = 20.0 | N_test = 500

--- [1/2] Continuous Full Steering (K=999, alpha=20.0) ---


Continuous Full Steering: 100%|██████████| 500/500 [3:08:58<00:00, 22.68s/it]



--- [2/2] Hard Cutoff Early-Stop (K=16, alpha=20.0) ---


Hard Cutoff Early-Stop: 100%|██████████| 500/500 [3:07:39<00:00, 22.52s/it]



✅ Part 1 (Continuous Full Steering, Hard Cutoff Early-Stop) completed for Alpha = 20.0!


In [7]:
# Cell 7: Compute BERTScore
print('\n' + '='*70)
print('BERTSCORE COMPUTATION')
print('='*70)
try:
    from bert_score import score as bert_score_fn
    for cond, results in results_dict.items():
        refs = [r['reference'] for r in results]
        hyps = [r['generated'] for r in results]
        print(f'  Computing BERTScore for {cond}...')
        P, R, F1 = bert_score_fn(hyps, refs, model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device=device)
        for r, bs in zip(results, F1.tolist()):
            r['bertscore_f1'] = bs
    print('✅ BERTScore computation complete!')
except Exception as e:
    print(f'⚠️ BERTScore computation skipped/failed: {e}')


BERTSCORE COMPUTATION
  Computing BERTScore for continuous...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing BERTScore for hard_cutoff...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERTScore computation complete!


In [8]:
# Cell 8: Summary & Export
suffix = '3'
part_num = 1
print('\n' + '='*85)
print(f'📊 PHASE 6A{suffix} (PART {part_num}): FAIR ABLATION SUMMARY (Alpha = {ALPHA_VAL})')
print('='*85)

def fmt(results):
    return {
        'alpha': ALPHA_VAL,
        'part': part_num,
        'rouge_l': np.mean([r['rouge_l'] for r in results]),
        'bertscore': np.mean([r.get('bertscore_f1', 0) for r in results]),
        'rep4': np.mean([r['rep_4gram'] for r in results]),
        'eos': np.mean([r['hit_eos'] for r in results]) * 100,
        'len': np.mean([r['num_tokens'] for r in results]),
        'lat_ms': np.mean([r['elapsed_ms'] for r in results])
    }

print(f'{"Schedule":<25} {"ROUGE-L%":>9} {"BERTSc":>7} {"Rep4":>6} {"EOS%":>5} {"Len":>5} {"ms":>7}')
print('-'*70)
summary_data = []
for key, res in results_dict.items():
    s = fmt(res)
    s['schedule'] = key
    summary_data.append(s)
    print(f'{key:<25} {s["rouge_l"]:>8.2f}% {s["bertscore"]:>6.4f} {s["rep4"]:>5.4f} {s["eos"]:>4.1f} {s["len"]:>5.1f} {s["lat_ms"]:>6.0f}')

out_json = os.path.join(OUTPUT_DIR, f'phase6a{suffix}_part{part_num}_alpha{int(ALPHA_VAL)}_results.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump({'alpha': ALPHA_VAL, 'part': part_num, 'n_test': len(test_subset), 'summary': summary_data,
               'detailed': {k: v for k, v in results_dict.items()}}, f, indent=2, ensure_ascii=False, default=str)

df_out = pd.DataFrame(summary_data)
out_csv = os.path.join(OUTPUT_DIR, f'phase6a{suffix}_part{part_num}_alpha{int(ALPHA_VAL)}_summary.csv')
df_out.to_csv(out_csv, index=False)

print(f'\n💾 Exported to {out_json} and {out_csv}')
print(f'🎉 PHASE 6A{suffix} (PART {part_num}) COMPLETED! (N_test = {len(test_subset)})')


📊 PHASE 6A3 (PART 1): FAIR ABLATION SUMMARY (Alpha = 20.0)
Schedule                   ROUGE-L%  BERTSc   Rep4  EOS%   Len      ms
----------------------------------------------------------------------
continuous                   23.37% 0.7134 0.0405  0.0 200.0  22670
hard_cutoff                  23.17% 0.7139 0.0387  0.0 200.0  22512

💾 Exported to /kaggle/working/phase6a3_part1_alpha20_results.json and /kaggle/working/phase6a3_part1_alpha20_summary.csv
🎉 PHASE 6A3 (PART 1) COMPLETED! (N_test = 500)
